In [1]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5-5"

In [3]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

In [4]:
message.content[0]

TextBlock(citations=None, text='Quantum computing is a type of computing that uses quantum-mechanical phenomena like superposition and entanglement, with qubits that can represent multiple states at once, to solve certain complex problems much faster than classical computers can.', type='text')

Multi-message requests with system prompts

In [5]:
# Start with an empty message list
messages = []

system_prompt = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "output_config": {
            "effort": "medium"
        }   
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
        
    return next(block.text for block in message.content if block.type == "text")

# Add the initial user question
add_user_message(messages, "What is 2x + 3 = 4?")

# Get Claude's response
answer = chat(messages, system_prompt)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Hmm, do I need to subtract 3 from both sides?")

# Get the follow-up response with full context
final_answer = chat(messages, system_prompt)

messages.append(final_answer)

messages

[{'role': 'user', 'content': 'What is 2x + 3 = 4?'},
 {'role': 'assistant',
  'content': 'Good question! Let\'s work through it together. The goal is to get **x** alone on one side of the equation.\n\nRight now, x is being multiplied by 2, and then 3 is being added. To undo those operations, we work backward.\n\n**Step 1:** Look at the equation: 2x + 3 = 4\n\nWhat operation could you do to *both sides* to get rid of the "+ 3" on the left? Try it, and tell me what the equation looks like afterward.'},
 {'role': 'user', 'content': 'Hmm, do I need to subtract 3 from both sides?'},
 'Yes, subtracting 3 from both sides is the right move. Subtraction undoes the addition, and doing it to both sides keeps the equation balanced.\n\nGo ahead and carry it out:\n\n- Left side: 2x + 3 − 3 = ?\n- Right side: 4 − 3 = ?\n\nWhat does your equation look like now?']